# 02 — Advanced pipeline V2

## 1. Objectif V2
Construire un pipeline avancé pour **Kaggle House Prices** visant une amélioration nette de la baseline V1 : prétraitement métier, features fortes, correction des variables asymétriques, modèles régularisés/boostés et blending simple.

## 2. Rappel score V1
La V1 compare Ridge et RandomForest avec imputation générique et one-hot encoding. Le score CV log RMSE attendu pour une baseline simple se situe autour de `0.145`, selon les données et l'environnement.

In [1]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
elif not (PROJECT_DIR / "src").exists():
    PROJECT_DIR = Path("../projects/kaggle-house-prices").resolve()

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

PROJECT_DIR

PosixPath('/workspace/House-Prices-Advanced-Regression-Techniques/projects/kaggle-house-prices')

## 3. Chargement des données
Les fichiers Kaggle doivent être placés dans `projects/kaggle-house-prices/data/raw/`. Si les données sont absentes, `load_train_test()` lève une erreur claire.

In [2]:
from src.data import load_train_test

train_df, test_df = load_train_test()
print(train_df.shape, test_df.shape)
train_df.head()

(1460, 81) (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## 4. Analyse rapide des valeurs manquantes métier
On inspecte les colonnes où `NaN` signifie souvent l'absence d'un équipement : garage, sous-sol, piscine, clôture, etc.

In [3]:
missing_summary = (
    train_df.isna().sum()
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
    .to_frame("missing_train")
)
missing_summary["missing_test"] = test_df.isna().sum().reindex(missing_summary.index).fillna(0).astype(int)
missing_summary.head(30)

,missing_train,missing_test
PoolQC,1453,1456
MiscFeature,1406,1408
Alley,1369,1352
Fence,1179,1169
MasVnrType,872,894
FireplaceQu,690,730
LotFrontage,259,227
GarageType,81,76
GarageYrBlt,81,78
GarageFinish,81,78


## 5. Feature engineering
La fonction V2 ajoute des surfaces totales, âges, indicateurs binaires et interactions entre qualité globale et dimensions du bien.

In [4]:
from src.advanced_features import prepare_advanced_features

X_train, X_test, y = prepare_advanced_features(train_df, test_df)
print(X_train.shape, X_test.shape, y.shape)
X_train.filter(regex="TotalSF|TotalBathrooms|HouseAge|OverallQual", axis=1).head()

(1460, 254) (1459, 254) (1460,)


,OverallQual,TotalSF,TotalBathrooms,HouseAge,OverallQual_TotalSF,OverallQual_GrLivArea,OverallQual_GarageArea,OverallQual_TotalBathrooms
0,7,7.850493,3.5,5,9.796069,9.390242,8.252446,3.238678
1,6,7.833996,2.5,31,9.625426,8.932345,7.923348,2.772589
2,7,7.903596,3.5,7,9.849190,9.433724,8.356320,3.238678
3,7,7.813592,2.0,91,9.759155,9.394327,8.410721,2.708050
4,8,8.114923,3.5,8,10.194103,9.774802,8.808220,3.367296


## 6. Transformation skew/log1p
Le skew est calculé sur les lignes train uniquement. Les colonnes numériques avec skew `> 0.75` sont transformées par `log1p`, en excluant les variables binaires et ordinales encodées manuellement.

In [5]:
skewed_cols = X_train.attrs.get("skewed_log1p_columns", [])
print(f"{len(skewed_cols)} colonnes transformées")
skewed_cols

28 colonnes transformées


['MiscVal',
 'PoolArea',
 'LotArea',
 '3SsnPorch',
 'LowQualFinSF',
 'KitchenAbvGr',
 'BsmtFinSF2',
 'ScreenPorch',
 'BsmtHalfBath',
 'EnclosedPorch',
 'MasVnrArea',
 'OverallQual_TotalSF',
 'LotFrontage',
 'OpenPorchSF',
 'OverallQual_GrLivArea',
 'TotalPorchSF',
 'TotalSF',
 'BsmtFinSF1',
 'WoodDeckSF',
 'TotalBsmtSF',
 'MSSubClass',
 '1stFlrSF',
 'GrLivArea',
 'TotalFlrSF',
 'OverallQual_GarageArea',
 'BsmtUnfSF',
 '2ndFlrSF',
 'OverallQual_TotalBathrooms']

## 7. Comparaison modèles avancés
Modèles obligatoires : Ridge, Lasso, ElasticNet, GradientBoostingRegressor et HistGradientBoostingRegressor. XGBoost/LightGBM sont ajoutés automatiquement s'ils sont installés.

In [6]:
from src.advanced_model import evaluate_advanced_models

scores = evaluate_advanced_models(X_train, y)
scores

XGBoost is not installed; skipping optional xgboost model.
LightGBM is not installed; skipping optional lightgbm model.
Evaluating ridge...


Evaluating lasso...


Evaluating elasticnet...


Evaluating gradient_boosting...


Evaluating hist_gradient_boosting...


,model,cv_rmse_log
0,lasso,0.126678
1,elasticnet,0.126809
2,gradient_boosting,0.128875
3,ridge,0.130698
4,hist_gradient_boosting,0.133612


## 8. Sélection du meilleur modèle
Le meilleur modèle single est celui avec le RMSE CV log le plus faible.

In [7]:
best_name = scores.iloc[0]["model"]
best_score = scores.iloc[0]["cv_rmse_log"]
print(f"Best single model: {best_name} — CV RMSE log: {best_score:.5f}")

Best single model: lasso — CV RMSE log: 0.12668


## 9. Blending
Un blending pondéré simple est utilisé. Les poids sont normalisés automatiquement et adaptés si XGBoost/LightGBM sont disponibles.

In [8]:
from src.advanced_model import fit_models_by_names, recommended_blend_weights, blend_predictions

weights = recommended_blend_weights(scores["model"].tolist())
model_names = [name for name in weights if name in set(scores["model"])]
fitted_models = fit_models_by_names(model_names, X_train, y)
blend_log_predictions = blend_predictions(fitted_models, X_test, weights)
print(weights)
blend_log_predictions[:5]

XGBoost is not installed; skipping optional xgboost model.
LightGBM is not installed; skipping optional lightgbm model.
Fitting elasticnet...


Fitting lasso...


Fitting ridge...
Fitting gradient_boosting...


Fitting hist_gradient_boosting...


{'elasticnet': 0.25, 'lasso': 0.2, 'ridge': 0.1, 'gradient_boosting': 0.25, 'hist_gradient_boosting': 0.2}


array([11.67919448, 11.97332387, 12.10712757, 12.17571469, 12.18397264])

## 10. Génération des fichiers de soumission
Le script complet génère deux CSV : meilleur modèle single et blend.

In [9]:
from src.submit_advanced import create_advanced_submissions

# Décommenter pour entraîner et écrire les CSV depuis le notebook.
# scores, best_submission, blend_submission = create_advanced_submissions()

## 11. Comparaison V1/V2
Comparer le tableau V2 ci-dessus au score V1 de référence. À compléter après exécution locale avec les données Kaggle.

In [10]:
comparison = scores.copy()
comparison.loc[len(comparison)] = {"model": "baseline_v1_reference", "cv_rmse_log": 0.145}
comparison.sort_values("cv_rmse_log")

,model,cv_rmse_log
0,lasso,0.126678
1,elasticnet,0.126809
2,gradient_boosting,0.128875
3,ridge,0.130698
4,hist_gradient_boosting,0.133612
5,baseline_v1_reference,0.145000


## 12. Limites
- Hyperparamètres initiaux non optimisés par recherche bayésienne.
- Blending simple sans méta-modèle.
- Pas encore de nettoyage spécifique des outliers influents.
- Pas d'analyse SHAP/permutation importance.

## 13. Prochaines itérations
- Ajouter un nettoyage contrôlé des outliers de `GrLivArea`.
- Tuner alpha/l1_ratio et paramètres des modèles boostés.
- Tester un stacking out-of-fold.
- Ajouter des groupes de features par quartier et qualité.
- Renseigner le score Kaggle public après upload.